# 05 Stage-1 localization recall — box-quality leading indicator (Phase 1a)

**Evaluation only.** Originally built for the two-stage line (Phase 1a/1b); now also the
**pre-registered leading indicator for V15 (NWD box loss)** — see
`docs/small_object_box_quality_notes.md`. The NWD thesis is that V15 *tightens the loose tiny boxes*
that capped V6. The most direct test is **class-agnostic localization recall@IoU0.5** on the small
Caries: if the boxes got tighter, V15's recall@IoU0.5 should rise vs V6 **before** the aggregate mAP
moves.

This notebook now runs **every detector it finds side-by-side** (auto-detects `V6` = `version6...`
and `V15` = `version15...`) and prints a **V6-vs-V15 recall comparison with per-class deltas**.

- **Phase 1a — the headline here (cheap, no retraining):** for each detector, how many GT lesions
  does it actually *localize* (class-agnostic box recall) at a few conf thresholds and box-IoU
  {0.30, 0.50}? Watch **Caries 1/2/3/5 recall@IoU0.5** — that is the NWD leading indicator. The
  small-Caries operating point is **conf≈0.05** (Phase 1a found recall collapses at higher conf).
- **Phase 1b — transfer check (OPTIONAL, OFF by default):** feed a detector's boxes into the
  Phase-0 `stage2_best.pt`. This belongs to the (closed) two-stage line and needs `stage2_best.pt`
  as an extra input; it is **not** relevant to the V15 box-quality question. Set `RUN_PHASE1B=True`
  only if you want it.

**Inputs (add as Kaggle Datasets/Models):** the **V15** detector (`version15...best.pt`) and,
for the comparison, the **V6** detector (`version6...best.pt`). `stage2_best.pt` is only needed if
`RUN_PHASE1B=True`. All are auto-detected by filename.


## 1. Setup

In [ ]:
# Ultralytics (Stage 1 = V6) + smp (Stage 2). Training kernels usually have Internet on.
try:
    import ultralytics, segmentation_models_pytorch  # noqa
    print("ultralytics + smp already present")
except ImportError:
    import subprocess, sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q",
                           "ultralytics", "segmentation-models-pytorch"])

In [ ]:
import os, json, math
from pathlib import Path
import numpy as np
import cv2
import yaml
import torch
import segmentation_models_pytorch as smp
from ultralytics import YOLO

cv2.setNumThreads(0)          # avoid cv2-thread contention
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", DEVICE)

## 2. Configuration

`ROI_INPUT` / `PAD_MODE` / `PAD_FACTOR` **must match the Phase-0 run that produced `stage2_best.pt`**
(otherwise the ROI distribution shifts and the transfer check is unfair). Phase 0 used
relative padding 1.5x at 224.

In [ ]:
# --- What to run ---
RUN_PHASE1B = False             # Phase 1a (recall) is the V15 leading indicator and needs no
                                # Stage 2. Phase 1b (Stage-2 transfer) belongs to the closed
                                # two-stage line and needs stage2_best.pt as an extra input.

# --- ROI framing (MUST match the stage2_best.pt training run; only used by Phase 1b) ---
ROI_INPUT   = 224
PAD_MODE    = "relative"
PAD_FACTOR  = 1.5
PAD_ABS_PX  = 48
ENCODER     = "resnet18"        # architecture of stage2_best.pt

# --- Stage 1 (detector) inference ---
IMG_SIZE_DET   = 768            # V6 / V15 were trained at 768
CAPTURE_CONF   = 0.01           # run once at very low conf, then filter in python
DET_IOU        = 0.60           # YOLO NMS IoU
MAX_DET        = 300

# --- Phase 1a recall sweep ---
CONF_SWEEP   = [0.05, 0.10, 0.25]   # Stage-1 conf thresholds to report recall at
RECALL_IOUS  = [0.30, 0.50]         # box-IoU for "localized"; 0.3 is usable because ROI is padded
LEAD_CONF    = 0.05                 # small-Caries operating point -> the V6-vs-V15 comparison conf

# --- Phase 1b pipeline mAP (only if RUN_PHASE1B) ---
PIPE_CONFS   = [0.05, 0.25]         # conf thresholds to score the full pipeline at
IOU_THRESHOLDS = np.linspace(0.5, 0.95, 10)   # mask-AP thresholds (same as src/03/04)

IMAGENET_MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
IMAGENET_STD  = np.array([0.229, 0.224, 0.225], dtype=np.float32)

# V6 per-class Mask mAP50-95 reference (src/03) + Phase-0 oracle reference, for context.
V6_REF_AP = {"abrasion":0.6471,"crown":0.6313,"filling":0.2799,"caries 1":0.1195,
             "caries 2":0.0845,"caries 3":0.0116,"caries 4":0.0000,"caries 5":0.1097,"caries 6":0.0051}
V6_REF_MAP = 0.2099
ORACLE_REF_MAP = 0.312    # Phase-0 oracle aggregate (context only)

def _norm_class_key(nm):
    s = str(nm).lower().replace("class", "")
    return " ".join(s.split())

print("RUN_PHASE1B", RUN_PHASE1B, "| ROI", ROI_INPUT, PAD_MODE, PAD_FACTOR,
      "| det imgsz", IMG_SIZE_DET, "| conf sweep", CONF_SWEEP, "| lead conf", LEAD_CONF)

## 3. Dataset + validation ground truth

In [ ]:
INPUT_ROOT = Path("/kaggle/input")
yc = list(INPUT_ROOT.rglob("yolo_seg_train.yaml"))
if not yc:
    raise FileNotFoundError("No yolo_seg_train.yaml under /kaggle/input.")
DATA_YAML = yc[0]
dcfg = yaml.safe_load(open(DATA_YAML, encoding="utf-8"))
names = dcfg.get("names")
if isinstance(names, dict):
    names = [names[k] for k in sorted(names)]
num_classes = len(names)
dataset_root = DATA_YAML.parent
yaml_root = Path(dcfg["path"]) if dcfg.get("path") else dataset_root
if not yaml_root.is_absolute():
    yaml_root = dataset_root / yaml_root

def resolve_split(v):
    if v is None: return None
    p = Path(v)
    if p.is_absolute(): return p
    for c in (yaml_root / p, dataset_root / p):
        if c.exists(): return c
    return yaml_root / p

val_images = resolve_split(dcfg.get("val", dcfg.get("valid")))
def labels_dir_for(images_dir):
    parts = list(Path(images_dir).parts)
    if "images" in parts:
        i = len(parts) - 1 - parts[::-1].index("images"); parts[i] = "labels"
        return Path(*parts)
    return Path(images_dir).parent / "labels"

IMG_EXTS = {".jpg",".jpeg",".png",".bmp",".webp",".tif",".tiff"}
def parse_seg_line(line):
    parts = line.strip().split()
    if len(parts) < 7: return None
    try:
        cls = int(float(parts[0])); coords = [float(v) for v in parts[1:]]
    except ValueError:
        return None
    if len(coords) % 2: coords = coords[:-1]
    poly = np.asarray(coords, dtype=np.float64).reshape(-1, 2)
    return (cls, poly) if len(poly) >= 3 else None

val_objs = {}   # image_path -> list of (cls, poly_norm)
lbl_dir = labels_dir_for(val_images)
for ip in sorted(p for p in Path(val_images).iterdir() if p.suffix.lower() in IMG_EXTS):
    lp = lbl_dir / (ip.stem + ".txt")
    objs = []
    if lp.exists():
        for line in lp.read_text().splitlines():
            pr = parse_seg_line(line)
            if pr is not None: objs.append(pr)
    val_objs[str(ip)] = objs

n_gt = np.zeros(num_classes, dtype=int)
for objs in val_objs.values():
    for c, _ in objs: n_gt[c] += 1
print("classes:", names)
print("val images:", len(val_objs), "| total GT objects:", int(n_gt.sum()))
for c in range(num_classes):
    print(f"  {names[c]:>14s}: n_gt={int(n_gt[c])}")

## 4. Load the detectors (V6 and/or V15) — and Stage 2 only if `RUN_PHASE1B`

Auto-detects every detector checkpoint under `/kaggle/input`: `V6` (`version6...`) and `V15`
(`version15...` / `nwd`). Whatever is present is run side-by-side in Phase 1a. For the V15-vs-V6
box-quality comparison you want **both** added as inputs. `stage2_best.pt` is loaded only when
`RUN_PHASE1B=True`.

In [ ]:
# Optional hard overrides — set to exact /kaggle/input paths if auto-detection picks wrong files.
# MANUAL_DETECTORS maps a label -> .pt path, e.g.
#   MANUAL_DETECTORS = {"V6": "/kaggle/input/v6/version6_best.pt",
#                       "V15": "/kaggle/input/v15/version15_best.pt"}
MANUAL_DETECTORS = {}
MANUAL_S2_PATH   = None

def find_pt(include, exclude=()):
    cands = []
    for p in Path("/kaggle/input").rglob("*.pt"):
        t = str(p).lower()
        if any(x in t for x in exclude): continue
        score = sum(10 for k in include if k in t) + (20 if p.name.lower().endswith("best.pt") else 0)
        if score > 0: cands.append((score, p))
    cands.sort(key=lambda z: z[0], reverse=True)
    return [p for _, p in cands]

# Build the detector set. V6 = "version6"/"v6"; V15 = "version15"/"v15"/"nwd". stage2 excluded so
# it never collides; cross-exclude version15/version6 keywords so the two detectors don't poach.
if MANUAL_DETECTORS:
    DETECTORS = {k: Path(v) for k, v in MANUAL_DETECTORS.items()}
else:
    DETECTORS = {}
    v6  = find_pt(["version6", "v6"],          exclude=["stage2", "version15", "version14"])
    v15 = find_pt(["version15", "v15", "nwd"], exclude=["stage2", "version6"])
    if v6:  DETECTORS["V6"]  = v6[0]
    if v15: DETECTORS["V15"] = v15[0]

if not DETECTORS:
    raise FileNotFoundError("No detector .pt found. Add version15_best.pt (and version6_best.pt for "
                            "the comparison) as Kaggle inputs, or set MANUAL_DETECTORS.")
print("Detectors to evaluate:")
for k, p in DETECTORS.items(): print(f"  {k}: {p}")
if "V15" not in DETECTORS:
    print("  WARNING: no V15 checkpoint found — the NWD leading indicator needs version15_best.pt.")
if "V6" not in DETECTORS:
    print("  WARNING: no V6 checkpoint found — without it there is no baseline to compare V15 against.")

detectors = {name: YOLO(str(p)) for name, p in DETECTORS.items()}

# Stage 2 (Phase 1b only).
stage2 = None
if RUN_PHASE1B:
    s2_cands = find_pt(["stage2"])
    print("\nStage2 candidates:"); [print("  ", p) for p in s2_cands[:5]]
    S2_PATH = Path(MANUAL_S2_PATH) if MANUAL_S2_PATH else (s2_cands[0] if s2_cands else None)
    if S2_PATH is None or not S2_PATH.exists():
        raise FileNotFoundError("RUN_PHASE1B=True but no stage2_best.pt found. Add it or set MANUAL_S2_PATH.")
    print("Using S2 :", S2_PATH)
    stage2 = smp.Unet(encoder_name=ENCODER, encoder_weights=None, in_channels=3,
                      classes=1, activation=None,
                      aux_params=dict(classes=num_classes, dropout=0.2)).to(DEVICE)
    state = torch.load(str(S2_PATH), map_location=DEVICE)
    state = state.get("state_dict", state) if isinstance(state, dict) else state
    missing, unexpected = stage2.load_state_dict(state, strict=False)
    stage2.eval()
    print("Stage2 loaded. missing:", len(missing), "unexpected:", len(unexpected))

## 5. Run each detector on the validation images (capture low-conf boxes once)

One pass per detector per image at `CAPTURE_CONF`; all boxes stored so the conf sweep is just a
Python filter (no re-inference). Boxes are used **class-agnostically** — only localization matters
for the recall gate. `det_by_image[name][img_path]` holds each detector's boxes.

In [ ]:
det_by_image = {name: {} for name in detectors}   # name -> path -> list of dict(box,conf,cls)
wh_by_image = {}
for ip in val_objs:
    img = cv2.imread(ip, cv2.IMREAD_COLOR)
    h, w = img.shape[:2]; wh_by_image[ip] = (w, h)
    for name, det in detectors.items():
        res = det.predict(source=img, imgsz=IMG_SIZE_DET, conf=CAPTURE_CONF, iou=DET_IOU,
                          max_det=MAX_DET, device=DEVICE, task="segment",
                          verbose=False, save=False)[0]
        dets = []
        if res.boxes is not None and len(res.boxes) > 0:
            xyxy = res.boxes.xyxy.cpu().numpy()
            conf = res.boxes.conf.cpu().numpy()
            cls  = res.boxes.cls.cpu().numpy().astype(int)
            for b, cf, cl in zip(xyxy, conf, cls):
                x0, y0, x1, y1 = [int(round(v)) for v in b]
                x0 = max(0, min(x0, w-1)); y0 = max(0, min(y0, h-1))
                x1 = max(x0+1, min(x1, w)); y1 = max(y0+1, min(y1, h))
                dets.append({"box": (x0, y0, x1, y1), "conf": float(cf), "cls": int(cl)})
        det_by_image[name][ip] = dets
for name in detectors:
    tot = sum(len(d) for d in det_by_image[name].values())
    print(f"{name}: captured {tot} boxes over {len(val_objs)} images at conf>={CAPTURE_CONF}")

## 6. Phase 1a — localization recall per detector + the V6-vs-V15 comparison (the headline)

For each GT object: is there a detector box (conf >= t) overlapping it by box-IoU >= thr?
Class-agnostic. Then the **V6-vs-V15 comparison** at `LEAD_CONF` (0.05): if NWD tightened the
boxes, **Caries 1/2/3/5 recall@IoU0.5** should be **higher for V15** (recall@IoU0.3 may be flat —
the point of NWD is the *tight* boxes, i.e. the IoU0.5 column).

In [ ]:
def gt_pixel_bbox(poly_norm, w, h):
    xs = poly_norm[:, 0] * w; ys = poly_norm[:, 1] * h
    return (max(0, int(np.floor(xs.min()))), max(0, int(np.floor(ys.min()))),
            min(w, int(np.ceil(xs.max()))), min(h, int(np.ceil(ys.max()))))

def box_iou(a, b):
    ix0, iy0 = max(a[0], b[0]), max(a[1], b[1])
    ix1, iy1 = min(a[2], b[2]), min(a[3], b[3])
    iw, ih = max(0, ix1 - ix0), max(0, iy1 - iy0)
    inter = iw * ih
    if inter <= 0: return 0.0
    aa = (a[2]-a[0])*(a[3]-a[1]); bb = (b[2]-b[0])*(b[3]-b[1])
    return inter / (aa + bb - inter)

import pandas as pd
recall_rows = []
for det_name in det_by_image:
    for iou_thr in RECALL_IOUS:
        for conf_t in CONF_SWEEP:
            hit = np.zeros(num_classes, dtype=int)
            for ip, objs in val_objs.items():
                w, h = wh_by_image[ip]
                boxes = [d["box"] for d in det_by_image[det_name][ip] if d["conf"] >= conf_t]
                for cls, poly in objs:
                    gb = gt_pixel_bbox(poly, w, h)
                    if any(box_iou(gb, bx) >= iou_thr for bx in boxes):
                        hit[cls] += 1
            for c in range(num_classes):
                recall_rows.append({"detector": det_name, "iou": iou_thr, "conf": conf_t,
                                    "class": names[c], "n_gt": int(n_gt[c]),
                                    "recall": (hit[c] / n_gt[c]) if n_gt[c] else float("nan")})
recall_df = pd.DataFrame(recall_rows)

# Per-detector recall tables at box-IoU 0.5 (the standard gate) and 0.30 (looser).
for det_name in det_by_image:
    sub = recall_df[recall_df.detector == det_name]
    print(f"=== {det_name}: localization recall @ box-IoU 0.5 ===")
    print(sub[sub.iou == 0.5].pivot(index="class", columns="conf", values="recall")
          .reindex(names).round(3).to_string())
    print(f"--- {det_name}: @ box-IoU 0.30 ---")
    print(sub[sub.iou == 0.30].pivot(index="class", columns="conf", values="recall")
          .reindex(names).round(3).to_string())
    print()

# Headline: V6-vs-V15 comparison at LEAD_CONF, both IoU gates side by side with deltas.
if "V6" in det_by_image and "V15" in det_by_image:
    for iou_thr in RECALL_IOUS:
        sub = recall_df[(recall_df.iou == iou_thr) & (recall_df.conf == LEAD_CONF)]
        cmp = sub.pivot(index="class", columns="detector", values="recall").reindex(names)
        cmp["delta(V15-V6)"] = cmp["V15"] - cmp["V6"]
        print(f"=== NWD leading indicator: V6 vs V15 recall @ box-IoU {iou_thr}, conf {LEAD_CONF} ===")
        print(cmp.round(3).to_string())
        caries = [c for c in names if _norm_class_key(c) in ("caries 1","caries 2","caries 3","caries 5")]
        md = cmp.loc[caries, "delta(V15-V6)"].mean()
        print(f"  -> mean delta over supported Caries (1/2/3/5): {md:+.3f}")
        print("     (the IoU0.5 row is the box-tightness signal NWD is meant to move)\n")
else:
    print("Need BOTH V6 and V15 checkpoints for the comparison table; only:", list(det_by_image))

## 7. Phase 1b — transfer check (OPTIONAL; `RUN_PHASE1B`, closed two-stage line)

Skipped unless `RUN_PHASE1B=True`. Feeds **one** detector's boxes (`PHASE1B_DET`, default `V15` if
present) into the Phase-0 `stage2_best.pt` and scores the full pipeline. Not relevant to the V15
box-quality question — kept for the two-stage record. `full` = all boxes (FPs included; no
background class). `TP-only` = boxes matching a GT (perfect FP rejection). Score = det conf x
Stage-2 class prob.

In [ ]:
def padded_square_box(x0, y0, x1, y1, w, h):
    bw, bh = x1 - x0, y1 - y0
    cx, cy = (x0 + x1) / 2.0, (y0 + y1) / 2.0
    half = (max(bw, bh) * (1.0 + PAD_FACTOR) / 2.0) if PAD_MODE == "relative" else (max(bw, bh) / 2.0 + PAD_ABS_PX)
    half = max(half, 1.0)
    sx0, sy0, sx1, sy1 = cx - half, cy - half, cx + half, cy + half
    sx0 = max(0.0, min(sx0, w - 1)); sy0 = max(0.0, min(sy0, h - 1))
    sx1 = max(sx0 + 1, min(sx1, w)); sy1 = max(sy0 + 1, min(sy1, h))
    return int(round(sx0)), int(round(sy0)), int(round(sx1)), int(round(sy1))

def gt_local_mask(poly_norm, w, h):
    pts = poly_norm.copy(); pts[:, 0] *= w; pts[:, 1] *= h
    gx0 = max(0, int(np.floor(pts[:,0].min()))); gy0 = max(0, int(np.floor(pts[:,1].min())))
    gx1 = min(w, int(np.ceil(pts[:,0].max())));  gy1 = min(h, int(np.ceil(pts[:,1].max())))
    gw = max(1, gx1 - gx0); gh = max(1, gy1 - gy0)
    m = np.zeros((gh, gw), dtype=np.uint8)
    pts[:, 0] -= gx0; pts[:, 1] -= gy0
    cv2.fillPoly(m, [pts.astype(np.int32)], 1)
    return (gx0, gy0, gx1, gy1), m

def iou_local(pbox, pmask, gbox, gmask):
    pa = int(pmask.sum()); ga = int(gmask.sum())
    if pa == 0 or ga == 0: return 0.0
    ix0 = max(pbox[0], gbox[0]); iy0 = max(pbox[1], gbox[1])
    ix1 = min(pbox[2], gbox[2]); iy1 = min(pbox[3], gbox[3])
    inter = 0
    if ix1 > ix0 and iy1 > iy0:
        pc = pmask[iy0-pbox[1]:iy1-pbox[1], ix0-pbox[0]:ix1-pbox[0]]
        gc = gmask[iy0-gbox[1]:iy1-gbox[1], ix0-gbox[0]:ix1-gbox[0]]
        inter = int(np.logical_and(pc, gc).sum())
    union = pa + ga - inter
    return inter / union if union > 0 else 0.0

@torch.no_grad()
def stage2_on_boxes(img, boxes):
    if not boxes: return []
    h, w = img.shape[:2]; xs = []; pboxes = []
    for (x0, y0, x1, y1) in boxes:
        sb = padded_square_box(x0, y0, x1, y1, w, h); pboxes.append(sb)
        crop = cv2.resize(img[sb[1]:sb[3], sb[0]:sb[2]], (ROI_INPUT, ROI_INPUT))
        x = cv2.cvtColor(crop, cv2.COLOR_BGR2RGB).astype(np.float32) / 255.0
        xs.append(((x - IMAGENET_MEAN) / IMAGENET_STD).transpose(2, 0, 1))
    batch = torch.from_numpy(np.stack(xs)).to(DEVICE)
    mlog, clog = stage2(batch)
    prob = torch.softmax(clog, 1).cpu().numpy()
    masks = (torch.sigmoid(mlog)[:, 0] > 0.5).cpu().numpy().astype(np.uint8)
    out = []
    for i, sb in enumerate(pboxes):
        pm = cv2.resize(masks[i], (sb[2]-sb[0], sb[3]-sb[1]), interpolation=cv2.INTER_NEAREST)
        out.append((sb, int(prob[i].argmax()), float(prob[i].max()), pm))
    return out

def ap_101(confs, tps, npos):
    if npos == 0: return float("nan")
    if not confs: return 0.0
    o = np.argsort(-np.asarray(confs)); tps = np.asarray(tps)[o]
    tp_c = np.cumsum(tps); fp_c = np.cumsum(1 - tps)
    rec = tp_c / npos; prec = tp_c / np.maximum(tp_c + fp_c, 1e-9)
    return sum((prec[rec >= r].max() if np.any(rec >= r) else 0.0) for r in np.linspace(0,1,101)) / 101.0

def pipeline_map(conf_t, tp_only=False):
    records = {(c, ti): [] for c in range(num_classes) for ti in range(len(IOU_THRESHOLDS))}
    for ip, objs in val_objs.items():
        w, h = wh_by_image[ip]
        gts = [(c, *gt_local_mask(p, w, h)) for c, p in objs]
        preds = []
        for it in cache[ip]:
            if it["v6_conf"] < conf_t: continue
            if tp_only and not it["matches_gt"]: continue
            preds.append((it["s2_cls"], it["v6_conf"] * it["s2_prob"], it["pbox"], it["pmask"]))
        for ti, thr in enumerate(IOU_THRESHOLDS):
            order = sorted(range(len(preds)), key=lambda i: preds[i][1], reverse=True)
            used = [False] * len(gts)
            for pi in order:
                pc, sc, pbox, pm = preds[pi]
                best, bg = 0.0, -1
                for gi, (gc, gbox, gm) in enumerate(gts):
                    if used[gi] or gc != pc: continue
                    v = iou_local(pbox, pm, gbox, gm)
                    if v > best: best, bg = v, gi
                tp = 1 if (bg >= 0 and best >= thr) else 0
                if tp: used[bg] = True
                records[(pc, ti)].append((sc, tp))
    pac = np.full(num_classes, np.nan)
    for c in range(num_classes):
        if n_gt[c] == 0: continue
        pac[c] = np.nanmean([ap_101([r[0] for r in records[(c,ti)]],
                                     [r[1] for r in records[(c,ti)]], n_gt[c])
                             for ti in range(len(IOU_THRESHOLDS))])
    return pac

# --- Execution (optional) ---
pipe_results = {}
if not RUN_PHASE1B:
    print("Phase 1b skipped (RUN_PHASE1B=False) — the V15 leading indicator is Phase 1a recall above.")
else:
    PHASE1B_DET = "V15" if "V15" in det_by_image else next(iter(det_by_image))
    print("Phase 1b on detector:", PHASE1B_DET)
    dsel = det_by_image[PHASE1B_DET]
    cache = {}   # path -> list of dict(pbox, s2_cls, s2_prob, pmask, v6_conf, matches_gt)
    for ip, objs in val_objs.items():
        img = cv2.imread(ip, cv2.IMREAD_COLOR); w, h = wh_by_image[ip]
        dets = dsel[ip]
        s2 = stage2_on_boxes(img, [d["box"] for d in dets])
        gt_boxes = [gt_pixel_bbox(p, w, h) for _, p in objs]
        items = []
        for d, (pbox, cls, prob, pm) in zip(dets, s2):
            matches = any(box_iou(d["box"], gb) >= 0.5 for gb in gt_boxes)
            items.append({"pbox": pbox, "s2_cls": cls, "s2_prob": prob, "pmask": pm,
                          "v6_conf": d["conf"], "matches_gt": matches})
        cache[ip] = items
    print("Stage-2 cached for", sum(len(v) for v in cache.values()), "boxes")

    for conf_t in PIPE_CONFS:
        pipe_results[f"full@{conf_t}"] = pipeline_map(conf_t, tp_only=False)
    pipe_results["TPonly@0.05"] = pipeline_map(0.05, tp_only=True)

    def show(tag, pac):
        valid = ~np.isnan(pac)
        print(f"\n=== {tag} === mAP50-95 = {np.nanmean(pac[valid]):.4f}  (V6 {V6_REF_MAP:.4f}, oracle {ORACLE_REF_MAP:.3f})")
        for c in range(num_classes):
            v6 = V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
            d = pac[c] - v6 if not (np.isnan(pac[c]) or np.isnan(v6)) else float("nan")
            print(f"  {names[c]:>14s} n={int(n_gt[c]):>3d}  pipe={pac[c]:.3f}  V6={v6:.3f}  d={d:+.3f}")
    for tag, pac in pipe_results.items():
        show(tag, pac)

## 8. Save outputs (Kaggle Save Version)

In [ ]:
OUT = Path("/kaggle/working")

# Phase 1a recall, all detectors (now carries a `detector` column).
recall_csv = OUT / "phase1a_recall.csv"
recall_df.to_csv(recall_csv, index=False)

# The headline artifact: V6-vs-V15 recall comparison at LEAD_CONF, per IoU gate, with deltas.
compare_path = OUT / "phase1a_recall_compare.csv"
if "V6" in det_by_image and "V15" in det_by_image:
    blocks = []
    for iou_thr in RECALL_IOUS:
        sub = recall_df[(recall_df.iou == iou_thr) & (recall_df.conf == LEAD_CONF)]
        cmp = sub.pivot(index="class", columns="detector", values="recall").reindex(names)
        cmp.insert(0, "iou", iou_thr)
        cmp["delta(V15-V6)"] = cmp["V15"] - cmp["V6"]
        blocks.append(cmp.reset_index())
    compare_df = pd.concat(blocks, ignore_index=True)
    compare_df.to_csv(compare_path, index=False)
    print("V6-vs-V15 recall comparison written:", compare_path)
else:
    compare_df = None
    print("Comparison CSV skipped (need both V6 and V15). Detectors:", list(det_by_image))

# Summary JSON: per-detector recall@IoU0.5, plus pipeline mAP if Phase 1b ran.
def _rec(det, t, c):
    m = recall_df[(recall_df.detector==det)&(recall_df.iou==0.5)&(recall_df.conf==t)&(recall_df["class"]==names[c])]
    if not len(m): return None
    r = m["recall"].values[0]
    return None if np.isnan(r) else round(float(r), 4)

summary = {
    "config": {"detectors": list(det_by_image), "RUN_PHASE1B": RUN_PHASE1B,
               "IMG_SIZE_DET": IMG_SIZE_DET, "CONF_SWEEP": CONF_SWEEP, "LEAD_CONF": LEAD_CONF},
    "stage1_recall_iou0.5": {
        det: {names[c]: {str(t): _rec(det, t, c) for t in CONF_SWEEP} for c in range(num_classes)}
        for det in det_by_image},
}
if RUN_PHASE1B and pipe_results:
    summary["pipeline_mAP"] = {tag: round(float(np.nanmean(pac[~np.isnan(pac)])), 4)
                               for tag, pac in pipe_results.items()}
    summary["v6_ref_mAP"] = V6_REF_MAP; summary["oracle_ref_mAP"] = ORACLE_REF_MAP
    prows = []
    for tag, pac in pipe_results.items():
        for c in range(num_classes):
            v6 = V6_REF_AP.get(_norm_class_key(names[c]), np.nan)
            prows.append({"variant": tag, "class": names[c], "n_gt": int(n_gt[c]),
                          "pipeline_AP": (None if np.isnan(pac[c]) else float(pac[c])),
                          "v6_ref_AP": (None if np.isnan(v6) else float(v6))})
    pd.DataFrame(prows).to_csv(OUT / "phase1b_pipeline.csv", index=False)

with open(OUT / "phase1_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print("\nSaved to /kaggle/working:")
for p in ["phase1a_recall.csv", "phase1a_recall_compare.csv", "phase1b_pipeline.csv", "phase1_summary.json"]:
    fp = OUT / p
    print(f"  [{'OK' if fp.exists() else '--'}] {p}")
if compare_df is not None:
    print("\n=== V6 vs V15 recall@IoU0.5, conf", LEAD_CONF, "===")
    print(compare_df[compare_df.iou == 0.5][["class","V6","V15","delta(V15-V6)"]].round(3).to_string(index=False))

## 9. How to read this → did NWD (V15) tighten the boxes?

**Look at the V6-vs-V15 comparison at box-IoU 0.5, conf 0.05 — the small Caries (1/2/3/5).**

- **`delta(V15-V6)` clearly positive on Caries 1/2/3/5 @IoU0.5** → NWD did its job: the tiny boxes
  got tighter, the leading indicator moved. This is the green light to keep going — sweep
  `NWD_CONSTANT` {3,5,8} (and λ {0.5,0.7}) in `src/08`, pruning by this same recall@IoU0.5, and
  re-check the aggregate mAP / per-class AP next.
- **delta ≈ 0 @IoU0.5** (and recall@IoU0.3 also flat) → this C/λ did **not** tighten the boxes.
  That is "this knob setting underwhelmed," not "NWD is dead" — try other C values before judging,
  then escalate to the assigner-level NWD / RFLA (the stronger lever in the research note).
- **delta negative** → NWD at this setting is hurting localization; back off (raise λ toward 1.0
  = more CIoU, or raise C).

The **@IoU0.3** columns are a sanity check: NWD targets *tight* localization, so the IoU0.5 column
should move more than IoU0.3. If recall rose at IoU0.3 but not 0.5, the detector is finding the same
lesions but still not boxing them tightly — NWD only partly worked.

Caveat: recall@IoU0.5 is the **box-quality leading indicator**, not the final dev metric. Even a
clear recall win must still be confirmed on the per-class Mask mAP (Caries up, large classes hold)
before V15 is called a win over V6's 0.234. Phase 1b (if you enabled it) is the old two-stage
transfer check and is not part of this decision.